In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib
import copy

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import gpt

In [ ]:
importlib.reload(gpt)

In [ ]:
ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Paul_and_Sophie'
OUT_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\big_results'

## Run model on all

In [ ]:
pdf_paths = []
for current_dir, subdirs, files in os.walk(ROOT):
    for file in files:
        path = os.path.join(current_dir, file)
        if path.endswith('.pdf'):
            pdf_paths.append(path)

pdf_paths

In [ ]:
len(pdf_paths)

In [ ]:
paths_df = pd.read_excel(os.path.join(OUT_DIR, 'all_articles2.xlsx'))

In [ ]:
pdfs_df.head()

In [ ]:
new_paths = set(pdf_paths) - set(old_df.path)
len(new_paths)

In [ ]:
gpt.use_key(gpt.KFIR_KEY)
client = gpt.get_client()

In [ ]:
files_with_paths = []
for path in tqdm(new_paths):
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
        file=open(path, "rb"),
        purpose="user_data")})


In [ ]:
len(files_with_paths)

In [ ]:
system_prompt = f'''
You are a scientist. You will be given several PDF files of scientific articles. You need to decide whether to include each one of them in a study.

You will be given criteria to decide whether to include or to exclude.

For each article, please print the article's title, then your reasoning, then the decision whether to include or exclude, in the following format:

```
TITLE
<article's title>

AUTHORS
<authors>

DOI
<doi ID>

PMID
<pubmed_id>

REASONING
<reasoning why to include or to exclude>

DECISION
<one of INCLUDE or EXCLUDE>

TITLE
<next article's title>
```

To get the DOI ID, look for e.g., `doi:10.1016/j.vaccine.2014.05.058`, then print `10.1016/j.vaccine.2014.05.058` as the DOI ID. If there is no DOI ID, print `<missing>` as the ID.

If you find a PUBMED ID (aka PMID), include it under PMID. Otherwise, print `<missing>`.


etc. 
You must do this for **each** input.

Write out the decisions for all inputs **in the same order as they are given**.
'''

In [ ]:
def run_model_on_batch(batch):
    content = [
                {
                    "type": "input_text",
                    "text": gpt.PAUL_CRITERIA,
                },]
    for item in batch:
        content.append({ "type": "input_file", "file_id": item['file_object'].id,})
    response = client.responses.create(
        model=gpt.Models.MINI,
        input=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": content,
            }
        ]
    )
    return response.output_text

In [ ]:
OUTPUTS = None
TO_PROCESS = None

In [ ]:
def run_on_missing_items(items, batch_size):
    global OUTPUTS, TO_PROCESS
    to_process = [item for item in items if 'title' not in item]
    TO_PROCESS = to_process
    print(f'Running on {len(to_process)} articles')

    outputs = []
    batches = list(batched(to_process, batch_size))
    try:
        for batch in tqdm(batches):
            outputs.append(run_model_on_batch(batch))
    except BaseException as e:
        print(f'Error: {type(e)}, {e}')
        print('Current files:', *[item['path'] for item in batch], sep='\n')

    OUTPUTS = outputs
    missing_items = 0
    updated_items = 0

    for i, (output, batch) in enumerate(zip(outputs, batches)):
        try:
            res = parse_pdf_response(output)
        except Exception as e:
            print(f'Error in batch {i}: {e}')
            missing_items += len(batch)
            continue
        if len(res) != len(batch):
            print(f'Batch {i}: Expected {len(batch)} results but got {len(res)}.')
            missing_items += len(batch)
            continue
        for item, result in zip(batch, res):
            item.update(result)
        updated_items += len(batch)

    print(f'Updated {updated_items} items, missing {missing_items} items.')


In [ ]:
DECISION_DICT = {'INCLUDE': True, 'EXCLUDE': False}
EXTRACTED_KEYS = ['title', 'reasoning', 'decision', 'authors', 'doi', 'pmid']
UPPER_CASE_KEYS = {key.upper() for key in EXTRACTED_KEYS}

def parse_pdf_response(response: str):
    items = []
    current_type = None

    for line in response.splitlines():
        line = line.strip()
        if not line or line in {'```', '---'}:
            continue
        if line in UPPER_CASE_KEYS:
            current_type = line
            if line == 'TITLE':
                items.append({key: '' for key in EXTRACTED_KEYS})
            continue

        if not current_type:
            raise ValueError(f'This line has no key to assign to: "{line}"')
        key = current_type.lower()
        _add_info(items[-1], key, line)

        if current_type in ('DECISION', 'DOI'):
            current_type = ''

    for i, item in enumerate(items):
        if not all(item[key] for key in EXTRACTED_KEYS):
            raise ValueError(f'Bad item {i}: {item}')
        decision = item['decision']
        if decision not in DECISION_DICT:
            raise ValueError(f'Invalid decision in item {i}: {decision}')
        item['decision'] = DECISION_DICT[decision]

    titles = [item['title'] for item in items]
    if len(set(titles)) < len(titles):
        raise ValueError(f'Titles are not unique: {titles}')

    return items

def _add_info(item, key, line):
    current = item[key]
    if current:
        item[key] = '\n'.join([current, line])
    else:
        item[key] = line

def clear_extracted(items):
    for item in items:
        for key in EXTRACTED_KEYS:
            if key in item:
                del item[key]

In [ ]:
run_on_missing_items(files_with_paths, 5)

In [ ]:
run_on_missing_items(files_with_paths, 3)

In [ ]:
run_on_missing_items(files_with_paths, 1)

doi_10.2340_00015555-3117_4a13b853d0.pdf was not a PDF

In [ ]:
df = pd.DataFrame(files_with_paths)
del df['file_object']
#df = df[~df.decision.isna()]
df.head()

In [ ]:
df.loc[df.pmid == '<missing>', 'pmid'] = np.nan
df.loc[df.doi == '<missing>', 'doi'] = np.nan
df.loc[df.pmid == '<misisng>', 'pmid'] = np.nan
df.loc[df.doi == '<misisng>', 'doi'] = np.nan
df.head()

In [ ]:
df['title_extract'] = df.title.apply(extract_letters)
len(df), df.title_extract.nunique(), df.doi.nunique()

In [ ]:
df.doi.isna().sum()

In [ ]:
df.columns

In [ ]:
old_df.columns

In [ ]:
new_df = df

In [ ]:
df = pd.concat([old_df.loc[:, new_df.columns], new_df], axis=0)
df.head()

## Duplicates and conflicts

In [ ]:
df['same_title_count'] = df.groupby('title_extract').title_extract.transform('count')
(df['same_title_count'] > 1).sum()

In [ ]:
df['same_doi'] = df.groupby('doi').doi.transform('count')
(df['same_doi'] > 1).sum()

In [ ]:
df['same_doi_and_title'] = df.groupby(['doi', 'title']).doi.transform('count')
(df['same_doi_and_title'] > 1).sum()

In [ ]:
title_counts_by_doi = df.groupby('doi').title_extract.nunique()
title_counts_by_doi = title_counts_by_doi[title_counts_by_doi > 1]
for doi in title_counts_by_doi.index:
    for item in df[df.doi == doi].path.tolist():
        print(item)
    print('---------------')

In [ ]:
decision_count_by_title = df.groupby('title_extract').decision.transform('nunique')
df['undecided_by_title'] = decision_count_by_title > 1
df['undecided_by_title'].sum()

In [ ]:
decision_count_by_doi = df.groupby('doi').decision.transform('nunique')
df['undecided_by_doi'] = decision_count_by_doi > 1
df['undecided_by_doi'].sum()

In [ ]:
df.sort_values(['doi', 'title_extract'], inplace=True)
df.head()

In [ ]:
df.to_excel(os.path.join(OUT_DIR, 'all_articles.xlsx'), index = False)

In [ ]:
## Big DF

In [ ]:
less_dups_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\SR_files_with_gpt_decision_2025-11-03-no_dups.xlsx")

In [ ]:
less_dups_df.tail()

In [ ]:
no_decision_df = less_dups_df[less_dups_df.old_decision.isna()]
len(no_decision_df)

In [ ]:
eligibility = gpt.check_eligibility(no_decision_df, criteria=gpt.PAUL_CRITERIA)
eligibility.head()

In [ ]:
del eligibility['old_decision']
eligibility.rename(columns={'decision': 'old_decision'}, inplace=True)
eligibility.head()

In [ ]:
old_less_dups_df = less_dups_df

In [ ]:
decision_df = old_less_dups_df[old_less_dups_df.old_decision.notna()]
less_dups_df = pd.concat([decision_df, eligibility], axis=0)
len(less_dups_df)

In [ ]:
less_dups_df['title_extract'] = less_dups_df.title.apply(extract_letters)

In [ ]:
no_dups_df = less_dups_df.drop_duplicates('title_extract')

In [ ]:
len(no_dups_df)

In [ ]:
no_dups_df.to_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\no_dups_with_paul_3_11.xlsx", index= False)

## Compare with big files

In [ ]:
df_300 = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\302_need_pdfs_Paul.xlsx")
df_300.head()

In [ ]:
df_300.columns

In [ ]:
huge_df = pd.read_excel(r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\SR_files_with_gpt_decision_2025-06-10.xlsx")
huge_df.head()

In [ ]:
huge_df.sort_values("is_in_paul", inplace = True)
huge_df['title_extract'] = huge_df.title.apply(extract_letters)
huge_df.drop_duplicates("title_extract", keep = "last", inplace = True)

In [ ]:
len(huge_df)

In [ ]:
huge_df.is_in_paul.sum()

In [ ]:
huge_df.rename(columns={'reasoning': 'old_reasoning'}, inplace=True)

In [ ]:
huge_df.columns

In [ ]:
no_decision_df = huge_df[huge_df.old_decision.isna()]
len(no_decision_df)

In [ ]:
eligibility = gpt.check_eligibility(no_decision_df, criteria=gpt.PAUL_CRITERIA)
eligibility.head()

In [ ]:
del eligibility['old_decision']
del eligibility['old_reasoning']
eligibility.rename(columns={'decision': 'old_decision', 'reasoning': 'old_reasoning'}, inplace=True)
eligibility.head()

In [ ]:
old_huge_df = huge_df

In [ ]:
decision_df = old_huge_df[old_huge_df.old_decision.notna()]
huge_df = pd.concat([decision_df, eligibility], axis=0)
len(huge_df)

In [ ]:
eligibility.columns

In [ ]:
len(huge_df), len(df_300), len(df)

In [ ]:
huge_df['can_find_doi'] = huge_df.DOI.isin(pdfs_df.doi) & huge_df.DOI.notna()
huge_df['can_find_title'] = huge_df.title_extract.isin(pdfs_df.title_extract)
huge_df['use_doi'] = huge_df.can_find_doi
huge_df['use_title'] = huge_df.can_find_title & ~huge_df.can_find_doi
(huge_df.use_doi | huge_df.use_title).sum()

In [ ]:
huge_df.DOI.nunique(), huge_df.DOI.notna().sum(), huge_df.title_extract.nunique()

In [ ]:
huge_df.DOI.notna().sum()

In [ ]:
huge_df.can_find_doi.sum(), huge_df.can_find_title.sum()

In [ ]:
usable_titles = huge_df.loc[huge_df.use_title, 'title_extract']
dois = huge_df.loc[huge_df.DOI.notna(), 'DOI']
pdfs_df['found_in_huge'] = pdfs_df.doi.isin(dois) | pdfs_df.title_extract.isin(usable_titles)
pdfs_df['found_in_huge'].sum()

In [ ]:
huge_df['same_title_count'] = huge_df.groupby('title_extract').title_extract.transform('count')
(huge_df['same_title_count'] > 1).sum()

In [ ]:
huge_df.sort_values('title_extract', inplace=True)

In [ ]:
pdfs_df['filename'] = pdfs_df.path.apply(os.path.basename)
pdfs_df.head()

In [ ]:
df.columns

In [ ]:
pdfs_df['undecided'] = pdfs_df.undecided_by_title | pdfs_df.undecided_by_doi

In [ ]:
df_for_joining = pdfs_df[['path', 'filename', 'title', 'title_extract', 'doi', 'reasoning', 'decision', 'undecided', 'found_in_huge']].copy()
decision_dict = {True: 'Include', False: 'Exclude'}
df_for_joining['decision1'] = df_for_joining['decision'].apply(decision_dict.get)
df_for_joining.loc[df_for_joining.undecided, 'decision1'] = 'Undecided'
df_for_joining.head()

In [ ]:
df_for_joining.groupby('decision1').decision1.count()

In [ ]:
df_for_joining.drop(columns=['decision', 'undecided'], inplace=True)
df_for_joining.columns

In [ ]:
df_for_joining.columns = ['pdf_path', 'pdf_file', 'title_from_pdf', 'title_extract', 'doi_from_pdf', 'reasoning', 'found_in_huge', 'decision']
df_for_joining

In [ ]:
df_for_joining_with_doi = df_for_joining[df_for_joining.doi_from_pdf.notna()].copy()
df_for_joining_with_doi.drop_duplicates('doi_from_pdf', inplace=True)
df_for_joining_with_doi.drop(columns=['title_extract', 'found_in_huge'], inplace=True)

In [ ]:
part_doi = pd.merge(huge_df[huge_df.use_doi], df_for_joining, left_on='DOI', right_on='doi_from_pdf', how='left')
part_doi.head()

In [ ]:
df_for_joining_with_title = df_for_joining.copy()
df_for_joining_with_title['has_doi'] = df_for_joining_with_title.doi_from_pdf.notna()
df_for_joining_with_title.sort_values('has_doi', inplace=True)
df_for_joining_with_title.drop_duplicates('title_extract', inplace=True, keep='last')
print(len(df_for_joining_with_title))
df_for_joining_with_title.head()

In [ ]:
df_for_joining_with_title.has_doi.sum()

In [ ]:
# df_for_joining_with_title.drop(columns=['has_doi', 'found_in_huge'], inplace=True)
part_title = pd.merge(huge_df[huge_df.use_title], df_for_joining, on='title_extract', how='left')
part_title.head()

In [ ]:
part_other = huge_df[~huge_df.use_title & ~huge_df.use_doi]
len(part_doi), len(part_title), len(part_other)

In [ ]:
not_in_huge = df_for_joining[~df_for_joining.found_in_huge].copy()

In [ ]:
final_huge_df = pd.concat([part_doi, part_title, part_other, not_in_huge], axis=0)
len(final_huge_df), len(huge_df)

In [ ]:
final_huge_df.columns

In [ ]:
final_huge_df.loc[final_huge_df.title.isna(), 'title'] = final_huge_df.loc[final_huge_df.title.isna(), 'title_from_pdf']

In [ ]:
final_huge_df.drop(columns=['title_extract_x', 'title_extract_y',  'can_find_doi', 'can_find_title', 'found_in_huge'], inplace=True)
final_huge_df['title_extract'] = final_huge_df.title.apply(extract_letters)

In [ ]:
final_huge_df.sort_values(['DOI', 'title_extract'], inplace=True)
# final_huge_df.drop(columns=['can_find_doi', 'can_find_title'], inplace=True)

In [ ]:
final_huge_df.to_excel(os.path.join(OUT_DIR, 'big_result_3_11.xlsx'))

## Deduplicate

In [ ]:
final_huge_df_dedup = final_huge_df.copy()
final_huge_df_dedup['has_doi'] = final_huge_df_dedup.doi_from_pdf.notna()

In [ ]:
def is_below_100(filename):
    if pd.isna(filename):
        return False
    basename = filename.split('.')[0]
    try:
        n = int(basename)
    except ValueError:
        return False
    return n <= 100
final_huge_df_dedup['is_below_100'] = final_huge_df_dedup.pdf_file.apply(is_below_100)

In [ ]:
final_huge_df_dedup.sort_values(['is_below_100', 'has_doi'], inplace=True)
final_huge_df_dedup.drop_duplicates('title_extract', inplace=True, keep='last')
print(len(final_huge_df), len(final_huge_df_dedup))
final_huge_df_dedup.drop(columns=['is_below_100', 'has_doi'], inplace=True)

In [ ]:
final_huge_df_dedup.to_excel(os.path.join(OUT_DIR, 'big_result_dedup.xlsx'))

In [ ]:
df.columns

In [ ]:
df.drop(columns=['undecided', 'score'], inplace=True)

In [ ]:
df.to_excel(os.path.join(OUT_DIR, 'all_articles2.xlsx'))

In [ ]:
same_doi_different_titles = df.groupby('doi').title_extract.nunique()
same_doi_different_titles = same_doi_different_titles[same_doi_different_titles>1]
same_doi_different_titles

In [ ]:
duplicate_titles = df.groupby('title_extract').title_extract.count()
duplicate_titles = duplicate_titles[duplicate_titles>1]

In [ ]:
prefix = ROOT + "\\"
for doi in same_doi_different_titles.index:
    for path in df.loc[df.doi == doi, 'path']:
        print(path.removeprefix(prefix))
    print()

for title in duplicate_titles.index:
    for path in df.loc[df.title_extract == title, 'path']:
        print(path.removeprefix(prefix))
    print()
